In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as st

: 

# Stat I project - FIFA World Cup 2022 player performance

By Macéo Drouot and Axel Bachon

Data: StatsBomb Open Data (men's World Cup 2022, Qatar)

## 1. Loading the data

In [ ]:
df = pd.read_csv('statsbomb_world_cup_2022_player_match.csv')
n = len(df)
df

In [ ]:
df.info()

In [ ]:
df.nunique()

The file has 1,995 rows and 38 columns, with no missing values. One row is one player in one match (64 matches, 32 teams, 680 players). Substitutes who did not come on are not in the file, so every row is a player who actually played (minimum 1.3 minutes).

In [ ]:
df.describe()

## 2. Data description

### Population under study

- Population: the appearances of players at the 2022 World Cup (one player in one match where he played)
- Number of observations: 1,995
- Time series (Y/N): N

### Preparing the variables

**position**: the file has 23 detailed positions (Left Back, Right Wing, Center Defensive Midfield, ...). That's too many for a table or a graph, so we group them into 4 roles:

- Goalkeeper
- Defender: all the Back positions (center backs, full backs and wing backs)
- Midfielder: all the Midfield positions
- Forward: wingers and forwards

In [ ]:
groups = {}
for p in df.position.unique():
    if p == 'Goalkeeper':
        groups[p] = 'Goalkeeper'
    elif 'Back' in p:
        groups[p] = 'Defender'
    elif 'Wing' in p or 'Forward' in p:
        groups[p] = 'Forward'
    else:
        groups[p] = 'Midfielder'

df['role'] = [groups[p] for p in df.position]
groups

**total_carry_distance**: the distance a player moved with the ball (sum of all his carries). StatsBomb gives it in pitch units, where the pitch is 120 x 80 (this corresponds to yards). We convert it to meters (1 yard = 0.9144 m).

In [ ]:
df['carry_m'] = df['total_carry_distance'] * 0.9144
df[['shots', 'carry_m', 'minutes_played']].describe()

### Variables

We chose 3 variables for the project:

| Name | Cat./Disc./Cont. | Obs./Exp. | levels, values or range |
|---|---|---|---|
| role (from position) | Categorical | Observed | Goalkeeper, Defender, Midfielder, Forward |
| shots | Discrete | Observed | 0 to 9 |
| carry_m (total_carry_distance) | Continuous | Observed | 0 to 1,024.8 meters |

Some other variables of the file: minutes_played (continuous, 1.3 to 141.4 min), team_name (categorical, 32 teams), stage (categorical, 6 stages from Group Stage to Final).

### Source and quality

- source: StatsBomb Open Data (https://github.com/statsbomb/open-data), competition 43, season 106. The CSV was built from the event and lineup files of the 64 matches.
- data collection method: census (all the players who played in all the matches of the tournament)
- quality/reliability: StatsBomb is a professional data provider and the definitions are documented. The file was checked (the goals give back the real score of all 64 matches, no duplicates, no missing values). Small limits: the position is only the starting position (changes during the match are not recorded), and the carry distance is a straight line between the start and the end of each carry, not the real distance run.

### Raw/processed

Processed: the CSV is built from the raw StatsBomb event files (counts and sums per player and match). We added two columns: `role` (grouped positions) and `carry_m` (carry distance in meters). No values were removed.

## 3. Frequency tables

### role (categorical)

In [ ]:
values_r, counts_r = np.unique(df.role, return_counts=True)
table_r = pd.DataFrame({'freq.': counts_r}, index=values_r)
table_r['rel. freq.'] = counts_r / n
table_r

Defenders (34%) and midfielders (32%) are the most common, then forwards (28%). Goalkeepers are only 7% because there is only one per team. No cumulated frequency because there is no order between roles.

### shots (discrete)

In [ ]:
values_s, counts_s = np.unique(df.shots, return_counts=True)
table_s = pd.DataFrame({'freq.': counts_s}, index=values_s)
table_s['rel. freq.'] = counts_s / n
table_s['cumul. rel. freq.'] = np.cumsum(table_s['rel. freq.'])
table_s

58% of the players don't shoot at all in a match and 83% shoot at most once. 5 shots or more is very rare (less than 2%). Nobody had 8 shots, and the maximum is 9.

### carry_m (continuous)

We use classes of 50 m between 0 and 400 m. Above 400 m there are few players, so we use bigger classes (400-500, 500-600 and 600-1100) to have at least 5 players in each class.

In [ ]:
bins_c = [0, 50, 100, 150, 200, 250, 300, 350, 400, 500, 600, 1100]
counts_c, bins_c = np.histogram(df.carry_m, bins=bins_c)
intervals = list(pd.IntervalIndex.from_breaks(bins_c, closed='left'))
intervals[-1] = pd.Interval(bins_c[-2], bins_c[-1], closed='both')  # last class is closed
table_c = pd.DataFrame({'freq.': counts_c}, index=pd.Index(intervals))
table_c['rel. freq.'] = counts_c / n
table_c['cumul. rel. freq.'] = np.cumsum(table_c['rel. freq.'])
table_c

Half of the players carry the ball less than about 110 m in a match, and 90% less than 300 m. Only a few players go above 500 m.

## 4. Graphs

### role

In [ ]:
counts = df.role.value_counts()
plt.bar(counts.index, counts.values / n)
plt.ylabel('relative frequencies')
plt.title('role of the players')
plt.show()

In [ ]:
plt.pie(counts.values, labels=counts.index, autopct='%.2f%%')
plt.title('role of the players')
plt.show()

Same result in both graphs: Defender > Midfielder > Forward > Goalkeeper.

### shots

In [ ]:
plt.bar(values_s, counts_s / n)
plt.xticks(values_s, values_s)
plt.xlabel('shots')
plt.ylabel('relative frequencies')
plt.title('shots per match')
plt.show()

In [ ]:
plt.plot(values_s, counts_s, marker='o')
plt.xticks(values_s, values_s)
plt.xlabel('shots')
plt.ylabel('frequencies')
plt.title('shots per match')
plt.show()

The distribution is very skewed to the right: most players have 0 shots in a match and the frequencies go down fast.

### carry_m

In [ ]:
plt.hist(df.carry_m, bins=bins_c, density=True)
plt.xlabel('carry distance (m)')
plt.ylabel('density')
plt.title('distance carried with the ball')
plt.show()

In [ ]:
plt.boxplot(df.carry_m)
plt.ylabel('carry distance (m)')
plt.title('distance carried with the ball')
plt.show()

The distribution is also skewed to the right: most players are between 0 and 200 m and there is a long tail to the right. The boxplot shows it too, with a median around 110 m and many points above the upper whisker.

## 5. Outliers

We use the 1.5 x IQR rule: a value is an outlier if it is below Q1 - 1.5 IQR or above Q3 + 1.5 IQR.

In [ ]:
for name in ['shots', 'carry_m']:
    q1, q3 = np.quantile(df[name], [.25, .75], method='inverted_cdf')
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    nb_out = ((df[name] < low) | (df[name] > high)).sum()
    print(name, ': limits', round(low, 1), round(high, 1), '-> outliers:', nb_out)

For shots, Q1 = 0 and Q3 = 1, so every player with 3 shots or more is an outlier (139 players, 7%). Let's see who they are:

In [ ]:
df[df.shots >= 3].role.value_counts() / len(df[df.shots >= 3])

Most of them are forwards (67%) and midfielders (27%), so it is normal that they shoot more. These are not errors, just attacking players having a good match.

For carry_m, 54 players are above the upper limit (about 420 m). Let's see who they are:

In [ ]:
out_c = df[df.carry_m > high]
print(out_c.role.value_counts() / len(out_c))
print('median minutes played:', out_c.minutes_played.median())

They are mostly defenders (61%) and midfielders (32%) who played the whole match (median about 101 min). Defenders often carry the ball forward from the back, so these values make sense. These are not errors either, so we keep all the outliers.

In [ ]:
table_moments = pd.DataFrame({'min': [min(df.shots), min(df.carry_m)],
                              'max': [max(df.shots), max(df.carry_m)],
                              'range': [max(df.shots) - min(df.shots), max(df.carry_m) - min(df.carry_m)],
                              'mean': [np.mean(df.shots), np.mean(df.carry_m)],
                              'variance': [np.var(df.shots), np.var(df.carry_m)],
                              'std': [np.std(df.shots), np.std(df.carry_m)],
                              'skewness': [st.skew(df.shots), st.skew(df.carry_m)],
                              'kurtosis': [st.kurtosis(df.shots), st.kurtosis(df.carry_m)]},
                             index=['shots', 'carry_m'])
table_moments

In [ ]:
q1, q2, q3 = np.quantile(df.carry_m, [.25, .5, .75], method='inverted_cdf') #def des quartiles
mean_c = np.mean(df.carry_m) # def de la mean de carry_m
dens_c = counts_c / (n * np.diff(bins_c)) # formule densité
i = np.argmax(dens_c) # va chercher l'indice 
mode_c = (bins_c[i] + bins_c[i + 1]) / 2
plt.hist(df.carry_m, bins=bins_c, density=True)
plt.axvline(q1, color='green', linestyle='--', label='Q1')
plt.axvline(q2, color='black', linestyle='--', label='median (Q2)')
plt.axvline(q3, color='green', linestyle=':', label='Q3')
plt.axvline(mean_c, color='red', label='mean')
plt.axvline(mode_c, color='orange', label='mode')
plt.xlabel('carry distance (m)')
plt.ylabel('density')
plt.title('distance carried with the ball')
plt.legend()
plt.show()